# Enhanced FlashGPT Turbo (flashgpt_turbo.ipynb)
## More advanced version with:
   - Memory-efficient chunked matrix multiplication
   - Multihead Latent Attention (MLA) module for improved reasoning
   - Built-in calculator tool using sympy
   - Enhanced wrapper architecture for improved capabilities

In [ ]:
import os
import time
import math
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm

# Setting up device and reproducibility
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Setup logging
logging.basicConfig(
    filename='training_log.txt',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger()


In [ ]:
# Helper function: Chunked matrix multiplication
def chunked_matmul(a, b, chunk_size=64):
    """
    Performs matrix multiplication of tensors a and b by splitting the last dimension
    of b into smaller chunks to reduce memory footprint and improve processing.
    a: Tensor of shape (..., M, K)
    b: Tensor of shape (..., K, N)
    chunk_size: Size of the chunk for processing
    Returns: Tensor of shape (..., M, N)
    """
    # Get the shape of b
    K = b.shape[-2]
    N = b.shape[-1]
    result_chunks = []
    for i in range(0, N, chunk_size):
        # b_chunk shape: (..., K, chunk_size)
        b_chunk = b[..., i:i+chunk_size]
        # Multiply a and b_chunk
        result_chunk = torch.matmul(a, b_chunk)
        result_chunks.append(result_chunk)
    # Concatenate along last dimension
    return torch.cat(result_chunks, dim=-1)


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import sympy as sp  # For the CalculatorTool

# --- Chunked Matmul for Memory Efficiency --- #
def chunked_matmul(a: torch.Tensor, b: torch.Tensor, chunk_size: int = 64) -> torch.Tensor:
    """
    Computes batched matrix multiplication between tensors 'a' and 'b' in smaller chunks.
    This helps reduce peak memory usage when computing attention scores.
    
    a: Tensor of shape (B, n_heads, T, head_dim)
    b: Tensor of shape (B, n_heads, head_dim, T)
    chunk_size: Number of tokens (T dimension) to process in one chunk.
    
    Returns:
        A tensor of shape (B, n_heads, T, T) computed by processing the T dimension in chunks.
        
    This technique is inspired by efficient attention approaches (e.g., FlashAttention)
    and related memory-reduction strategies from recent transformer research.
    """
    B, n_heads, T, head_dim = a.shape
    output_chunks = []
    # Process the T dimension of 'a' in chunks to avoid large intermediate tensors.
    for start in range(0, T, chunk_size):
        end = min(start + chunk_size, T)
        # Compute partial attention scores for the chunk
        chunk_result = torch.matmul(a[:, :, start:end, :], b)  # (B, n_heads, chunk_size, T)
        output_chunks.append(chunk_result)
    return torch.cat(output_chunks, dim=2)

# --- RMSNorm (improved version) --- #
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-4):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Compute the L2 norm along the last dimension
        norm_x = x.norm(2, dim=-1, keepdim=True)
        # Compute root mean square (rms) and normalize
        rms_x = norm_x / math.sqrt(x.shape[-1])
        return (x / (rms_x + self.eps)) * self.weight

# --- SwiGLU Feed-Forward Block --- #
class SwiGLU(nn.Module):
    def __init__(self, hidden_dim: int, expansion_factor: int = 4, dropout_prob: float = 0.1):
        super().__init__()
        self.expanded_dim = expansion_factor * hidden_dim
        self.fc_in = nn.Linear(hidden_dim, 2 * self.expanded_dim)
        self.fc_out = nn.Linear(self.expanded_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Linear projection splits into two parts
        x_proj = self.fc_in(x)
        x1, x2 = x_proj.chunk(2, dim=-1)
        # SwiGLU: Activation (SiLU) applied to one part and multiplied by the other
        x_out = F.silu(x1) * x2
        x_out = self.fc_out(x_out)
        return self.dropout(x_out)

# --- ALiBi Positional Bias --- #
def build_alibi_tensor(batch_size: int, n_heads: int, seq_len: int, device: torch.device) -> torch.Tensor:
    def get_slopes(n: int):
        def get_slopes_power_of_2(n: int):
            start = 2 ** (-8.0 / n)
            ratio = start
            return [start * (ratio ** i) for i in range(n)]
        if math.log2(n).is_integer():
            return get_slopes_power_of_2(n)
        else:
            closest_power_of_2 = 2 ** math.ceil(math.log2(n))
            slopes = get_slopes_power_of_2(closest_power_of_2)
            return slopes[:n]
    slopes = torch.tensor(get_slopes(n_heads), device=device).unsqueeze(-1).unsqueeze(-1)
    arange_tensor = torch.arange(seq_len, device=device).unsqueeze(0).unsqueeze(0)
    alibi = slopes * arange_tensor
    return alibi

# --- GPTConfig for configuration --- #
class GPTConfig:
    def __init__(self, vocab_size: int, max_seq_len: int, n_embd: int, n_layer: int, n_head: int, dropout_prob: float = 0.1, alibi: bool = True, flash_attention: bool = True):
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.dropout_prob = dropout_prob
        self.alibi = alibi
        self.flash_attention = flash_attention

# --- MultiHeadSelfAttention with Chunked Matmul --- #
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        assert self.n_embd % self.n_head == 0, "Embedding dim must be divisible by number of heads"
        self.head_dim = self.n_embd // self.n_head
        
        self.q_proj = nn.Linear(self.n_embd, self.n_embd)
        self.k_proj = nn.Linear(self.n_embd, self.n_embd)
        self.v_proj = nn.Linear(self.n_embd, self.n_embd)
        self.out_proj = nn.Linear(self.n_embd, self.n_embd)
        self.dropout = nn.Dropout(config.dropout_prob)
        self.flash_attention = config.flash_attention
        
        # Precompute a causal mask for self-attention (upper triangular is masked)
        self.register_buffer("causal_mask", torch.tril(torch.ones(config.max_seq_len, config.max_seq_len)), persistent=False)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, alibi_bias: torch.Tensor = None) -> torch.Tensor:
        B, T, C = x.size()
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # Reshape for multi-head attention
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if self.flash_attention and alibi_bias is None and attention_mask is None:
            # Leverage PyTorch’s efficient scaled dot-product implementation when conditions allow.
            attn_output = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout.p, is_causal=True)
        else:
            # Compute attention scores in chunks to be memory efficient.
            attn_scores = chunked_matmul(q, k.transpose(-2, -1), chunk_size=64) / math.sqrt(self.head_dim)
            # Apply the causal mask (ensures autoregressive property)
            causal_mask = self.causal_mask[:T, :T]
            attn_scores = attn_scores.masked_fill(causal_mask == 0, float('-inf'))
            if alibi_bias is not None:
                attn_scores = attn_scores + alibi_bias[:, :, :T]
            if attention_mask is not None:
                extended_mask = attention_mask.unsqueeze(1).unsqueeze(2)
                attn_scores = attn_scores.masked_fill(extended_mask == 0, float('-inf'))
            attn_weights = F.softmax(attn_scores, dim=-1)
            attn_weights = self.dropout(attn_weights)
            attn_output = torch.matmul(attn_weights, v)

        # Combine heads back together
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, C)
        output = self.out_proj(attn_output)
        return self.dropout(output)

# --- Transformer Block --- #
class TransformerBlock(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.attn_norm = RMSNorm(config.n_embd)
        self.ffn_norm = RMSNorm(config.n_embd)
        self.attn = MultiHeadSelfAttention(config)
        self.mlp = SwiGLU(config.n_embd, expansion_factor=4, dropout_prob=config.dropout_prob)
        self.dropout = nn.Dropout(config.dropout_prob)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, alibi_bias: torch.Tensor = None) -> torch.Tensor:
        normed_x = self.attn_norm(x)
        attn_out = self.attn(normed_x, attention_mask=attention_mask, alibi_bias=alibi_bias)
        x = x + attn_out
        normed_x2 = self.ffn_norm(x)
        ffn_out = self.mlp(normed_x2)
        return x + ffn_out

# --- Main GPT Model --- #
class GPTModel(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.vocab_size = config.vocab_size
        self.max_seq_len = config.max_seq_len
        self.n_embd = config.n_embd
        self.n_layer = config.n_layer
        self.n_head = config.n_head
        
        self.wte = nn.Embedding(self.vocab_size, self.n_embd)
        self.drop = nn.Dropout(config.dropout_prob)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(self.n_layer)])
        self.norm_f = RMSNorm(config.n_embd)
        self.apply(self._init_weights)

    def _init_weights(self, module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0, std=0.02)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        B, T = input_ids.shape
        assert T <= self.max_seq_len, "Sequence length exceeds model's maximum."
        token_embeddings = self.wte(input_ids)
        x = self.drop(token_embeddings)
        alibi_bias = None
        if self.config.alibi:
            alibi_bias = build_alibi_tensor(B, self.n_head, T, device=x.device)
        for block in self.blocks:
            x = block(x, attention_mask=attention_mask, alibi_bias=alibi_bias)
        return self.norm_f(x)

# --- GPT LM Head (ties LM head with embeddings) --- #
class GPTLMHeadModel(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.transformer = GPTModel(config)
        self.vocab_size = config.vocab_size
        self.n_embd = config.n_embd

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None, labels: torch.Tensor = None) -> dict:
        hidden_states = self.transformer(input_ids, attention_mask=attention_mask)
        logits = F.linear(hidden_states, self.transformer.wte.weight)
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = nn.CrossEntropyLoss()(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        return {"loss": loss, "logits": logits}


# =============================================================================
#                      New Additions for Enhanced Efficiency
# =============================================================================

# --- MLA (Multihead Latent Attention) Module --- #
class MLA(nn.Module):
    """
    Multihead Latent Attention (MLA) aggregates latent representations using a set of learnable latent tokens.
    This module is designed to boost the model's capacity for reasoning, math, and dense information encoding,
    drawing inspiration from recent research on latent transformers.
    """
    def __init__(self, n_embd: int, n_latent: int, n_head: int, dropout_prob: float = 0.1):
        super().__init__()
        self.n_latent = n_latent
        # Learnable latent tokens
        self.latent = nn.Parameter(torch.randn(n_latent, n_embd))
        self.q_proj = nn.Linear(n_embd, n_embd)
        self.k_proj = nn.Linear(n_embd, n_embd)
        self.v_proj = nn.Linear(n_embd, n_embd)
        self.out_proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout_prob)
        self.n_head = n_head
        self.head_dim = n_embd // n_head

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, n_embd)
        B, T, C = x.size()
        # Expand latent tokens for each batch: (B, n_latent, n_embd)
        latent = self.latent.unsqueeze(0).expand(B, self.n_latent, C)
        # Compute queries from latent tokens
        q = self.q_proj(latent)  # (B, n_latent, n_embd)
        # Compute keys and values from input tokens
        k = self.k_proj(x)       # (B, T, n_embd)
        v = self.v_proj(x)       # (B, T, n_embd)
        # Reshape for multi-head attention
        q = q.view(B, self.n_latent, self.n_head, self.head_dim).transpose(1, 2)  # (B, n_head, n_latent, head_dim)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)                # (B, n_head, T, head_dim)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)                # (B, n_head, T, head_dim)
        # Compute attention scores in latent space
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, n_head, n_latent, T)
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        latent_out = torch.matmul(attn_weights, v)  # (B, n_head, n_latent, head_dim)
        latent_out = latent_out.transpose(1, 2).contiguous().view(B, self.n_latent, C)  # (B, n_latent, n_embd)
        # Aggregate latent outputs (here we simply average)
        aggregated = latent_out.mean(dim=1)  # (B, n_embd)
        # Reinject the aggregated latent representation into each token
        enhanced = x + self.out_proj(aggregated).unsqueeze(1).expand(-1, T, -1)
        return enhanced


# --- CalculatorTool: Built-In Math Calculator --- #
class CalculatorTool:
    """
    A lightweight symbolic math calculator using sympy.
    This tool enables the model to perform inbuilt mathematical evaluations and simplifications.
    """
    def __init__(self):
        pass

    def calculate(self, expression: str) -> str:
        """
        Evaluates a mathematical expression using sympy.
        Returns the simplified result as a string.
        """
        try:
            expr = sp.sympify(expression)
            result = sp.nsimplify(expr)
            return str(result)
        except Exception as e:
            return f"Error: {e}"


# --- Enhanced GPT LM Head Model Wrapper --- #
class EnhancedGPTLMHeadModel(nn.Module):
    """
    A wrapper that integrates the original GPTLMHeadModel with additional modules for
    improved reasoning, math, and dense information encoding.
    
    - When 'use_mla' is enabled, applies MLA (Multihead Latent Attention) to the hidden states.
    - When 'use_calculator' is enabled, provides an inbuilt calculator tool for mathematical evaluations.
    """
    def __init__(self, config: GPTConfig, mla_n_latent: int = 16, use_mla: bool = True, use_calculator: bool = False):
        super().__init__()
        self.gpt_lm_head = GPTLMHeadModel(config)
        self.use_mla = use_mla
        if self.use_mla:
            self.mla = MLA(config.n_embd, n_latent=mla_n_latent, n_head=config.n_head, dropout_prob=config.dropout_prob)
        self.use_calculator = use_calculator
        if self.use_calculator:
            self.calculator = CalculatorTool()

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None, labels: torch.Tensor = None) -> dict:
        # Get the base output from the original model
        output = self.gpt_lm_head(input_ids, attention_mask=attention_mask, labels=labels)
        # Obtain hidden states from the transformer for further enhancement
        hidden_states = self.gpt_lm_head.transformer(input_ids, attention_mask=attention_mask)
        hidden_states = self.gpt_lm_head.transformer.norm_f(hidden_states)
        # If MLA is enabled, refine the hidden states with latent attention
        if self.use_mla:
            hidden_states = self.mla(hidden_states)
            # Recompute logits using the enhanced hidden states
            logits = F.linear(hidden_states, self.gpt_lm_head.transformer.wte.weight)
            output["logits"] = logits
        return output

    def calculate(self, expression: str) -> str:
        """
        Uses the inbuilt CalculatorTool to evaluate a mathematical expression.
        """
        if self.use_calculator:
            return self.calculator.calculate(expression)
        else:
            return "Calculator tool not enabled."


In [ ]:
# from transformers import GPT2Tokenizer

# # Initialize tokenizer
# tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
# tokenizer.pad_token = tokenizer.eos_token

# print('Loading TinyStories dataset...')
# dataset = load_dataset('roneneldan/TinyStories')

# if 'validation' not in dataset:
#     dataset = dataset['train'].train_test_split(test_size=0.1)
#     train_dataset = dataset['train']
#     val_dataset = dataset['test']
# else:
#     train_dataset = dataset['train']
#     val_dataset = dataset['validation']

# def tokenize_function(examples):
#     return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

# print('Tokenizing training and validation datasets...')
# train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
# val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)

# train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])
# val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

# train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
# val_dataloader = DataLoader(val_dataset, batch_size=8)

# print('Data loaded and tokenized successfully.')
import os
import pandas as pd
from datasets import DatasetDict, load_dataset
from transformers import GPT2Tokenizer
from torch.utils.data import DataLoader

# Ensure Kaggle API credentials are set up correctly
dataset_name = 'shubchat/1002-short-stories-from-project-guttenberg'
os.system(f'kaggle datasets download -d {dataset_name} -p ./ --unzip')

# Step 1: Load the `stories.csv` dataset
data_files = {'train': 'stories.csv'}  # Ensure correct filename
raw_datasets = load_dataset('csv', data_files=data_files)

# Step 2: Split the dataset into train/validation (90% train, 10% validation)
split_datasets = raw_datasets['train'].train_test_split(test_size=0.1)
datasets = DatasetDict({
    'train': split_datasets['train'],
    'validation': split_datasets['test']
})

# Step 3: Initialize the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# Step 4: Tokenize the datasets
def tokenize_function(examples):
    return tokenizer(examples['content'], truncation=True, padding='max_length', max_length=128)

tokenized_datasets = datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=['bookno', 'content'],
    num_proc=4  # Adjust based on your CPU cores
)
# Step 5: Convert datasets to PyTorch tensors
tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask'])

# Step 6: Create DataLoaders
train_dataloader = DataLoader(tokenized_datasets['train'], batch_size=8, shuffle=True)
val_dataloader = DataLoader(tokenized_datasets['validation'], batch_size=8)

print('Data successfully loaded, tokenized, and prepared for training!')


In [ ]:
# Adjusted model configuration with MLA and optimization techniques
config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    max_seq_len=1024,  # Extended sequence length for improved context understanding
    n_embd=512,  # Increased embedding dimension for richer representations
    n_layer=10,  # Optimized number of layers to balance efficiency and performance
    n_head=8,  # More heads to improve reasoning and attention granularity
    dropout_prob=0.05,  # Slightly reduced dropout for better information retention
    alibi=True,  # Keeping ALiBi for better extrapolation
    flash_attention=True  # Utilizing FlashAttention for GPU speedup
)

# Initialize the enhanced model
model = EnhancedGPTLMHeadModel(
    config, 
    mla_n_latent=16,  # Number of latent tokens for Multihead Latent Attention
    use_mla=True,  # Enables MLA
    use_calculator=True  # Enables the built-in CalculatorTool
).to(device)

# Compile the model for CPU acceleration if not using CUDA
if device.type != 'cuda' and hasattr(torch, 'compile'):
    print("Compiling model for CPU acceleration using torch.compile (DeepSeek optimization activated)!")
    try:
        model = torch.compile(model, mode='reduce-overhead')
    except Exception as e:
        print("Compilation failed, continuing without compile. Error:", e)

# Print total parameter count
total_params = sum(p.numel() for p in model.parameters())
print(f'Total Parameters: {total_params}')
print('Enhanced Model initialized successfully.')


In [ ]:
from torch.optim import AdamW

# Optimizer and scheduler setup
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

num_epochs = 3
num_training_steps = len(train_dataloader) * num_epochs
num_warmup_steps = 0

def lr_lambda(current_step):
    if current_step < num_warmup_steps:
        return float(current_step) / float(max(1, num_warmup_steps))
    return max(0.0, float(num_training_steps - current_step) / float(max(1, num_training_steps - num_warmup_steps)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


In [ ]:
def load_latest_checkpoint(model, optimizer, scheduler, checkpoint_dir='./checkpoints_turbo'):
    if not os.path.exists(checkpoint_dir):
        print(f"No checkpoint directory found at '{checkpoint_dir}'. Starting training from scratch.")
        return model, optimizer, scheduler, 0, 0

    checkpoints = [os.path.join(checkpoint_dir, ckpt) for ckpt in os.listdir(checkpoint_dir) if ckpt.endswith('.pt')]
    if not checkpoints:
        print(f"No checkpoints found in '{checkpoint_dir}'. Starting training from scratch.")
        return model, optimizer, scheduler, 0, 0

    latest_ckpt = max(checkpoints, key=os.path.getctime)
    checkpoint = torch.load(latest_ckpt, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    try:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    except ValueError as e:
        print(f"Warning: Optimizer state dict mismatch - {e}. Skipping optimizer state load.")
    try:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    except ValueError as e:
        print(f"Warning: Scheduler state dict mismatch - {e}. Skipping scheduler state load.")
    epoch = checkpoint['epoch']
    global_step = checkpoint['global_step']
    print(f"Loaded checkpoint '{latest_ckpt}' from epoch {epoch+1}, step {global_step}.")
    return model, optimizer, scheduler, epoch + 1, global_step

model, optimizer, scheduler, start_epoch, global_step = load_latest_checkpoint(model, optimizer, scheduler)

In [ ]:
# Optimized Training Loop
epochs = 3
checkpoint_interval = 600  # seconds
start_epoch = 0
global_step = 0
last_checkpoint_time = time.time()
checkpoint_dir = './checkpoints_turbo'

print('Starting training...')
model.train()

for epoch in range(start_epoch, epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    epoch_iterator = tqdm(train_dataloader, desc="Training")
    for batch in epoch_iterator:
        inputs = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=inputs, attention_mask=attention_mask, labels=inputs)
        loss = outputs["loss"]

        # If loss is NaN, skip the update
        if torch.isnan(loss):
            print(f"NaN loss encountered at step {global_step}. Skipping update.")
            logger.warning(f"NaN loss at step {global_step}")
            optimizer.zero_grad()
            continue

        try:
            with torch.autograd.detect_anomaly():
                loss.backward()
        except RuntimeError as e:
            print(f"Runtime error during backward pass at step {global_step}: {e}")
            logger.error(f"Backward error at step {global_step}: {e}")
            optimizer.zero_grad()
            continue

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        global_step += 1
        epoch_iterator.set_postfix(loss=loss.item())

        if time.time() - last_checkpoint_time >= checkpoint_interval:
            os.makedirs(checkpoint_dir, exist_ok=True)
            ckpt_path = os.path.join(checkpoint_dir, f'checkpoint-epoch{epoch+1}-step{global_step}.pt')
            torch.save({
                'epoch': epoch,
                'global_step': global_step,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict()
            }, ckpt_path)
            print(f"Saved checkpoint at step {global_step}")
            logger.info(f"Saved checkpoint at step {global_step}")
            last_checkpoint_time = time.time()

    # Run validation at the end of each epoch
    model.eval()
    val_losses = []
    generated_outputs = []
    expected_outputs = []
    with torch.no_grad():
        for batch in val_dataloader:
            inputs = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=inputs, attention_mask=attention_mask, labels=inputs)
            val_losses.append(outputs["loss"].item())

            sample_input = inputs[0:1]
            generated_ids = sample_input
            for _ in range(50):
                logits = F.linear(model.transformer(generated_ids), model.transformer.wte.weight)
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                generated_ids = torch.cat((generated_ids, next_token), dim=1)
            generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
            expected_text = tokenizer.decode(inputs[0], skip_special_tokens=True)
            generated_outputs.append(generated_text)
            expected_outputs.append(expected_text)
    avg_val_loss = sum(val_losses) / len(val_losses)
    print(f"Validation Loss after epoch {epoch+1}: {avg_val_loss}")
    logger.info(f"Epoch {epoch+1} - Validation Loss: {avg_val_loss}")

    for i, (gen, exp) in enumerate(zip(generated_outputs[:3], expected_outputs[:3])):
        log_str = f"Sample {i+1}:\nExpected: {exp}\nGenerated: {gen}\n{'-'*20}"
        print(log_str)
        logger.info(log_str)

    model.train()

print('Training complete!')


In [ ]:
# Checkpoint Loading and Inference Demo
import torch
import torch.nn.functional as F

def generate_text(prompt, model, tokenizer, max_length=50, temperature=0.7, top_k=50):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    generated = input_ids
    with torch.no_grad():
        for _ in range(max_length - input_ids.size(1)):
            outputs = model(generated)
            logits = outputs['logits'][:, -1, :] / temperature
            # Top-k filtering
            values, _ = torch.topk(logits, k=top_k, dim=-1)
            min_values = values[:, -1].unsqueeze(1)
            filtered_logits = torch.where(logits < min_values, torch.full_like(logits, float('-inf')), logits)
            probabilities = F.softmax(filtered_logits, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)
            generated = torch.cat((generated, next_token), dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(generated[0], skip_special_tokens=True)

def load_latest_checkpoint_simple(model, optimizer, scheduler, checkpoint_dir='./checkpoints_turbo'):
    checkpoints = [os.path.join(checkpoint_dir, ckpt) for ckpt in os.listdir(checkpoint_dir) if ckpt.endswith('.pt')]
    if not checkpoints:
        print('No checkpoints found.')
        return 0
    latest_ckpt = max(checkpoints, key=os.path.getctime)
    checkpoint = torch.load(latest_ckpt, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    print(f"Loaded checkpoint from {latest_ckpt} at step {checkpoint['global_step']}")
    return checkpoint['global_step']

global_step = load_latest_checkpoint_simple(model, optimizer, scheduler)
prompt = "Once upon a time, there was a girl named Alice who discovered a mysterious key"
print("Prompt:", prompt)
print("Generated Text:", generate_text(prompt, model, tokenizer))

'''\nNo checkpoints found.\nPrompt: Once upon a time, there was a girl named Alice who discovered a mysterious key\nGenerated Text: Once upon a time, there was a girl named Alice who discovered a mysterious key. One day, she saw a kind girl named Lily. She loved to go outside and watch the house with her friends. \n\nThen, she found a big box of flowers. She asked her mom, \"Be the end, this?\" Her mom said, \"That was so proud of me to make it feel so she couldn\'t make us. \n\nLily decided to fix the car to the other dress and put on the stick to take it on. She put it on the sky, he saw a beautiful noise. But the\n'''